# refactored x-bte annotation: validating API yamls against schema

In [1]:
## CX: allows multiple lines of code to print from one code block
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import json
import jsonref
import jsonschema
from pathlib import Path
import yaml

## loading schema

In [2]:
## create and check that the desired path is generated
Path.home().joinpath("Desktop", "translator_extensions",
                     "x-bte", "DRAFT-refactor-x-bte-schema.json")

PosixPath('/Users/colleenxu/Desktop/translator_extensions/x-bte/DRAFT-refactor-x-bte-schema.json')

In [3]:
## use jsonref package to load json schema with references traversed/filled-in
schema_path = Path.home().joinpath("Desktop", "translator_extensions",
                                   "x-bte", "DRAFT-refactor-x-bte-schema.json")

with open(schema_path) as file:
    schema = jsonref.load(file)

In [4]:
## check that schema loaded
schema.keys()
schema["description"]

dict_keys(['$schema', 'title', 'description', 'type', 'properties', 'allOf', '$defs'])

'Manually-written schema for x-bte annotation refactor. Work-in-progress.'

In [5]:
## a check that $ref have been traversed/filled-in
schema["properties"]["components"] \
    ["properties"]["x-bte-annotations"] \
    ["patternProperties"]["^[A-Za-z0-9_-]+$"] \
    ["properties"]["inputs"] \
    ["properties"]["category"]

{'description': "This is the (semantic) type of biological entities represented by these IDs. The value must be a class (aka node category) in the biolink-model: it should be the most specific class that fits the most biological entities represented in the data. Avoid using classes labeled 'mixin' when possible. The format is PascalCase, no prefix. Validation is currently just on format, so it doesn't check whether the value actually exists in the biolink-model version the tool is using.",
 'type': 'string',
 'pattern': '^([A-Z][a-z]+)+$',
 '$comment': 'For full validation, this would need enum of acceptable biolink-model node categories (not mixins and must be bioentities?). And would need to be maintained, up-to-date with biolink-model version the tool is using.'}

In [6]:
## another check that $ref have been traversed/filled-in
schema["properties"]["components"] \
    ["properties"]["x-bte-annotations"] \
    ["patternProperties"]["^[A-Za-z0-9_-]+$"] \
    ["properties"]["provenance"] \
    ["oneOf"][2] \
    ["properties"]["mapping"]

{'description': "Maps response-field's values (must be strings) to structured provenance data",
 'type': 'object',
 'additionalProperties': False,
 'patternProperties': {'^.+$': {'description': 'This is the simplest case: all data retrieved using this annotation has the same, single set of provenance info.',
   'type': 'object',
   'unevaluatedProperties': False,
   '$comment': "Don't include any $ref in here.",
   'required': ['knowledge_level', 'agent_type'],
   'properties': {'knowledge_level': {'description': 'Biolink-model knowledge level for the data',
     '$comment': "Enum would need to be kept up-to-date with the tool's biolink-model version",
     'type': 'string',
     'enum': ['knowledge_assertion',
      'logical_entailment',
      'prediction',
      'statistical_association',
      'observation',
      'not_provided']},
    'agent_type': {'description': 'Biolink-model agent type for the data',
     '$comment': "Enum would need to be kept up-to-date with the tool's biolin

## checking a (refactored) SmartAPI yaml with x-bte annotation 

In [7]:
yaml_path = Path.home().joinpath("Desktop", "translator-api-registry", 
                                 "AGR", "agr.yaml")

with open(yaml_path) as file:
    smartapi_yaml = yaml.load(file, Loader=yaml.SafeLoader)
    smartapi_yaml = json.dumps(smartapi_yaml, indent=2)
    smartapi_yaml = jsonref.loads(smartapi_yaml)

In [8]:
smartapi_yaml['components']['x-bte-annotations'].keys()

dict_keys(['biomarker-via-orthology_gene_disease', 'biomarker-via-orthology_disease_gene'])

## validate example against schema

In [9]:
jsonschema.validate(instance=smartapi_yaml, schema=schema)
## YAYAYAYAYAY this means it validated!!!!!!!

### error: if a required field is missing

In [11]:
## but what if it's a fluke?? 
## remove a required key from one association 
removed1 = example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['predicateInfo'].pop('biolink')

## then try to validate, this 
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)
## so it works in catching the error yayyyyyyyy

ValidationError: 'biolink' is a required property

Failed validating 'required' in schema['properties']['components']['properties']['x-bte-association-retrieval']['patternProperties']['.']['properties']['predicateInfo']:
    {'additionalProperties': False,
     'properties': {'biolink': {'oneOf': [{'additionalProperties': False,
                                           'description': 'Info set in '
                                                          'registry, not '
                                                          'dependent on '
                                                          'API response. '
                                                          'Use the value '
                                                          'field',
                                           'properties': {'value': {'oneOf': [{'items': {'type': 'string'},
                                                                               'minItems': 1,
                            

In [12]:
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['predicateInfo']['biolink'] = removed1

In [13]:
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)
## and it's back and accepted again 

### error: if there are two mins in range

note: this test requires at least one numeric measure in the registry entry

In [14]:
## another check: look at the range for the first numericMeasure
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]['range']

{'minExclusive': 0, 'maxInclusive': 1}

In [15]:
## add an error: minInclusive
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]['range']['minInclusive'] = -1

## then try to validate, this 
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)

ValidationError: {'anyOf': [{'type': 'object', 'required': ['minExclusive', 'minInclusive']}, {'type': 'object', 'required': ['maxExclusive', 'maxInclusive']}]} is not allowed for {'minExclusive': 0, 'maxInclusive': 1, 'minInclusive': -1}

Failed validating 'not' in schema['properties']['components']['properties']['x-bte-association-retrieval']['patternProperties']['.']['properties']['numericMeasures']['items']['allOf'][1]['properties']['range']:
    {'additionalProperties': False,
     'description': 'Object, Python dict-like. Defines an expected lower '
                    'and/or upper bound for values (minimum and maximum). '
                    'Inclusive means the range includes the boundary '
                    'number; exclusive means the range does not. If the '
                    'actual lower-bound is negative-infinity and/or the '
                    'actual upper bound is positive-infinity, do not set a '
                    'bound',
     'minProperties': 1,
     'not': 

In [16]:
## fix the error and re-validate
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]['range'].pop('minInclusive')

-1

In [17]:
## then try to validate, this 
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)

### error: static publications typing

note: 
this test currently requires a numeric measure. 
be careful that you don't overwrite an existing publications key when doing this test

In [18]:
## make a publications/pmid key within measureReferences
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]['measureReferences']\
['publications'] = {"pmid": {}}  ## make it first 

example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]

{'name': 'DisGeNET gene-disease association score',
 'responseField': 'disgenet.genes_related_to_disease.score',
 'measureReferences': {'websites': {'value': 'https://www.disgenet.org/dbinfo#section31'},
  'publications': {'pmid': {}}},
 'range': {'minExclusive': 0, 'maxInclusive': 1},
 'directionMeaning': {'larger': 'more_evidence'}}

In [19]:
## then try to validate 
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)
## catches that there should be a key called value

ValidationError: 'value' is a required property

Failed validating 'required' in schema[0]:
    {'additionalProperties': False,
     'properties': {'value': {'oneOf': [{'items': {'type': ['string',
                                                            'number']},
                                         'minItems': 1,
                                         'type': 'array'},
                                        {'type': ['string', 'number']}]}},
     'required': ['value'],
     'type': 'object'}

On instance:
    {}

In [20]:
## remove the publications key
example_from_yaml['components']['x-bte-association-retrieval']['disease-gene1']\
['numericMeasures'][0]['measureReferences'].pop('publications')

{'pmid': {}}

In [21]:
## then try to validate 
jsonschema.validate(instance=example_from_yaml, schema=schema_from_yaml)
## catches that there should be a key called value

## Export JSON files for the yamls

In [ ]:
json_schema_path = pathlib.Path.cwd().joinpath("draft7_schema_registry.json")
with open(json_schema_path, "w") as file:
    json.dump(schema_from_yaml, file, indent=2)

In [ ]:
json_example_path = pathlib.Path.cwd().joinpath("draft7_registry_disgenetDG.json")
with open(json_example_path, "w") as file:
    json.dump(example_from_yaml, file, indent=2)